# 02 – Train DNN model (GTZAN)

This notebook **reads** `features.csv` and `labels.csv` produced by **notebook 01**, then fits PCA, trains the DNN, and saves `models/DNN3.weights.h5` and `models/pca`. Run notebook 01 first to generate the feature files.

In [1]:
import pathlib
import sys
from pathlib import Path

_root = Path.cwd() if (Path.cwd() / "music_genre_classifier").is_dir() else Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import os
import logging
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
logging.getLogger("tensorflow").setLevel(logging.ERROR)  # Suppress TF GPU-on-Windows warning

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from music_genre_classifier.config import N_CLASSES

MODELS_DIR = pathlib.Path(_root) / "models"
MODELS_DIR.mkdir(exist_ok=True)

In [ ]:
# Model definition (from music_genre_classifier.model)
from keras.layers import Dense, Dropout, Input
from keras.models import Sequential


def build_dnn_model(input_dim=215, n_classes=10):
    """Build the dense neural network used for genre classification."""
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    model.add(Dense(512, activation="relu"))
    model.add(Dropout(0.4))
    model.add(Dense(512, activation="relu"))
    model.add(Dropout(0.4))
    model.add(Dense(256, activation="relu"))
    model.add(Dropout(0.6))
    model.add(Dense(n_classes, activation="softmax"))
    model.compile(
        optimizer="nadam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def load_trained_model(weights_path="DNN3.weights.h5", input_dim=215, n_classes=10):
    """Build the DNN model and load pre-trained weights."""
    model = build_dnn_model(input_dim=input_dim, n_classes=n_classes)
    model.load_weights(weights_path)
    return model


def forward_pass(model, x):
    """Convenience helper for tests: run a forward pass."""
    return model.predict(x)

In [2]:
# Load features and labels produced by notebook 01
features_path = _root / "features.csv"
labels_path = _root / "labels.csv"
if not features_path.exists() or not labels_path.exists():
    raise FileNotFoundError(
        "Run notebook 01_data_exploration.ipynb first to generate features.csv and labels.csv"
    )

features = pd.read_csv(features_path, index_col=0, header=[0, 1, 2])
labels = pd.read_csv(labels_path)
# Align by path: ensure same order
labels = labels.set_index("path").reindex(features.index).dropna()
features = features.loc[labels.index]
# Drop samples with missing features (PCA does not accept NaN; e.g. failed extraction in notebook 01)
valid = ~features.isna().any(axis=1)
features = features.loc[valid]
labels = labels.loc[valid]
y = labels["label"].values.astype(np.int32)
X = features.values.astype("float32")
print(X.shape, y.shape)

(999, 518) (999,)


In [3]:
# Standardize and fit PCA similar to the original workflow.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=215)
X_pca = pca.fit_transform(X_scaled)

X_train, X_val, y_train, y_val = train_test_split(X_pca, y, test_size=0.2, random_state=42)
X_train.shape, X_val.shape

((799, 215), (200, 215))

In [6]:
n_features = X_train.shape[1]

from keras.layers import Dense, Dropout, Input
from keras.models import Sequential


def build_dnn_models(input_dim: int = 215, n_classes: int = 10) -> Sequential:
    """Build the dense neural network used for genre classification."""
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    model.add(Dense(512, activation="relu"))
    model.add(Dropout(0.4))
    model.add(Dense(512, activation="relu"))
    model.add(Dropout(0.4))
    model.add(Dense(256, activation="relu"))
    model.add(Dropout(0.6))
    model.add(Dense(n_classes, activation="softmax"))

    model.compile(
        optimizer="nadam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_dnn_models(input_dim=n_features, n_classes=N_CLASSES)
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 512)            │       110,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 507,146 (1.93 MB)

 Trainable params: 507,146 (1.93 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=64,
)

model.save_weights(MODELS_DIR / "DNN3.weights.h5")

import pickle
with open(MODELS_DIR / "pca", "wb") as f:
    pickle.dump(pca, f)

Epoch 1/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.1464 - loss: 2.7478 - val_accuracy: 0.2800 - val_loss: 1.9176
Epoch 2/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2954 - loss: 1.9911 - val_accuracy: 0.4450 - val_loss: 1.6357
Epoch 3/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.4581 - loss: 1.6073 - val_accuracy: 0.5500 - val_loss: 1.4051
Epoch 4/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5081 - loss: 1.4036 - val_accuracy: 0.5850 - val_loss: 1.2170
Epoch 5/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5757 - loss: 1.2199 - val_accuracy: 0.6450 - val_loss: 1.1076
Epoch 6/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6095 - loss: 1.0787 - val_accuracy: 0.7050 - val_loss: 0.9296
Epoch 7/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6809 - loss: 0.8999 - val_accuracy: 0.7150 - val_loss: 0.8470
Epoch 8/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7347 - loss: 0.7517 - val_accuracy: 0.7550 - v

ValueError: The filename must end in `.weights.h5`. Received: filepath=c:\Users\datae\Desktop\bin\github-revision\Deep-Learning_Music-style-detection\models\DNN3.h5